In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import math
import os
import random
import sys
import zipfile
import matplotlib.pyplot as plt
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from scipy.special import ndtri, ndtr
from scipy.stats import wasserstein_distance
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors

# Paths

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, "/content")
import utils as ut
import metrics as mt

TRAIN_ZIP = "/content/drive/MyDrive/cursach-hse/mnist_data/train.csv.zip"
AE_WEIGHTS = "/content/drive/MyDrive/cursach-hse/compare_two_unseen/ae_100dim_new.pth"
DDPM_UNET = "/content/drive/MyDrive/cursach-hse/ddpm_unet.pth"
CFM_WEIGHTS = "/content/drive/MyDrive/cursach-hse/compare_two_unseen/cfm_weights_new.pth"
OUT_DIR = Path("/content/phase_coverage_outputs")
OUT_DIR.mkdir(exist_ok=True)

ut.set_seed(42)
device = ut.get_device(prefer_cuda=True)
device

device(type='cuda')

## Load real data and autoencoder embeddings

In [ ]:
train_csv = ut.ensure_unzipped(TRAIN_ZIP, OUT_DIR / "data")
real_images, labels = ut.load_mnist_csv(train_csv, max_rows=42000, require_labels=True)
ae = ut.load_autoencoder(AE_WEIGHTS, device=device)
real_z = ut.embed_images(ae, real_images, device=device, batch_size=512)
real_z.shape, labels.shape

((42000, 100), (42000,))

## Controlled validation: matched, missing-mode, and skewed proxy generators

In [ ]:
def proxy_fake_from_real(
    real_z,
    labels,
    mode,
    sample_size=None,
    missing_labels=(0,),
    class_weights=None,
    seed=42,
):
    """
    mode='matched': sample according to empirical real distribution.
    mode='missing': sample only labels not in missing_labels.
    mode='skewed': sample labels using class_weights, e.g. {0: 5.0, 1: 0.2, ...}.
    """
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    n = len(real_z)
    sample_size = int(sample_size or n)

    if mode == "matched":
        p = np.ones(n, dtype=np.float64) / n
    elif mode == "missing":
        keep = ~np.isin(labels, list(missing_labels))
        p = keep.astype(np.float64)
        p /= p.sum()
    elif mode == "skewed":
        if class_weights is None:
            class_weights = {int(c): 1.0 for c in np.unique(labels)}
            class_weights[int(np.unique(labels)[0])] = 5.0
        p = np.array([class_weights.get(int(y), 1.0) for y in labels], dtype=np.float64)
        p /= p.sum()

    idx = rng.choice(n, size=sample_size, replace=True, p=p)
    return real_z[idx], labels[idx]


def run_stress_tests(
    real_z,
    labels,
    sample_size=None,
    k=20,
    gamma=1.0,
    tau=0.25,
    delta=math.log(2.0),
    seed=42,
):
    scenarios = []
    # Matched proxy
    fake_z, fake_y = proxy_fake_from_real(real_z, labels, "matched", sample_size, seed=seed)
    scenarios.append(("matched_proxy", fake_z, fake_y))
    # Missing one digit and missing several digits
    for missing in ([0], [0, 1], [8, 9]):
        fake_z, fake_y = proxy_fake_from_real(real_z, labels, "missing", sample_size, missing_labels=missing, seed=seed)
        scenarios.append((f"missing_{'_'.join(map(str, missing))}", fake_z, fake_y))
    # Skewed priors: overproduce digit 0, underproduce 1 and 2
    weights = {int(c): 1.0 for c in np.unique(labels)}
    weights[0] = 5.0
    weights[1] = 0.2
    weights[2] = 0.2
    fake_z, fake_y = proxy_fake_from_real(real_z, labels, "skewed", sample_size, class_weights=weights, seed=seed)
    scenarios.append(("skewed_0_high_1_2_low", fake_z, fake_y))

    rows = []
    for name, fz, fy in scenarios:
        res, details = mt.estimate_completeness_uniformity(
            real_z, fz, k=k, gamma=gamma, tau=tau, delta=delta, return_details=True
        )
        row = asdict(res)
        row["scenario"] = name
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
stress = run_stress_tests(real_z, labels, sample_size=len(real_z), k=20, gamma=1.0, tau=0.25)
stress

,completeness,uniformity,mean_abs_log_ratio,median_abs_log_ratio,covered_fraction,n_real,n_fake,k,gamma,tau,delta,scenario
0,0.999976,0.997476,0.180961,0.162519,0.999976,42000,42000,20,1.0,0.25,0.693147,matched_proxy
1,0.903381,0.996047,2.032575,0.223144,0.903381,42000,42000,20,1.0,0.25,0.693147,missing_0
2,0.791714,0.987880,4.259480,0.336472,0.791714,42000,42000,20,1.0,0.25,0.693147,missing_0_1
3,0.845690,0.967201,2.577677,0.336472,0.845690,42000,42000,20,1.0,0.25,0.693147,missing_8_9
4,0.858381,0.787557,0.804702,0.356675,0.858381,42000,42000,20,1.0,0.25,0.693147,skewed_0_high_1_2_low


## Evaluate generated trained models

DDIM

In [ ]:
fake_arr = np.load("/content/drive/MyDrive/cursach-hse/ddim_imgs_42000.npy")
fake_z = ut.embed_images(ae, fake_arr, device=device)
metrics, details = mt.estimate_completeness_uniformity(real_z, fake_z, k=20, gamma=1.0, tau=0.25, return_details=True)
# details.head()

In [ ]:
metrics

CoverageResult(completeness=0.8121666666666667, uniformity=0.5869367652663364, mean_abs_log_ratio=1.0913099293917137, median_abs_log_ratio=0.798507693651105, covered_fraction=0.8121666666666667, n_real=42000, n_fake=42000, k=20, gamma=1.0, tau=0.25, delta=0.6931471805599453)

In [ ]:
ut.save_local_ratio_plot(details, OUT_DIR / f"ddim_local_ratios.png", title=f"ddim: local mass-ratio")

## Sample-size curves

In [ ]:
def sample_size_curve(
    real_z,
    fake_z,
    sizes=(500, 1000, 2000, 5000, 10000),
    repeats=5,
    seed=42,
    **metric_kwargs,
):
    rng = np.random.default_rng(seed)
    rows = []
    for size in sizes:
        size_eff = min(int(size), len(fake_z))
        for rep in range(repeats):
            idx = rng.choice(len(fake_z), size=size_eff, replace=False)
            res = mt.estimate_completeness_uniformity(real_z, fake_z[idx], **metric_kwargs)
            row = asdict(res)
            row.update({"sample_size": size_eff, "repeat": rep})
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
curve = sample_size_curve(real_z, fake_z, sizes=[1000, 5000, 10000, 25000, 42000], repeats=5, k=20, gamma=1.0, tau=0.25)
curve.groupby("sample_size")[["completeness", "uniformity"]].mean()

,completeness,uniformity
sample_size,,
1000,0.220671,0.000000
5000,0.650476,0.479590
10000,0.622781,0.658595
25000,0.805029,0.602422
42000,0.812167,0.586937


In [ ]:
curve.groupby("sample_size")[["completeness", "uniformity"]].std()

,completeness,uniformity
sample_size,,
1000,0.006879,0.000000
5000,0.004763,0.004824
10000,0.004702,0.003691
25000,0.002550,0.001976
42000,0.000000,0.000000


In [ ]:
ut.save_curve_plot(curve, OUT_DIR / f"ddim_sample_size_curve.png")

CFM

In [ ]:
fake_arr = np.load("/content/drive/MyDrive/cursach-hse/cfm_imgs_42000.npy")
fake_z = ut.embed_images(ae, fake_arr, device=device)
metrics, details = mt.estimate_completeness_uniformity(real_z, fake_z, k=20, gamma=1.0, tau=0.25, return_details=True)
details.head()

,rho,radius,real_count,fake_count,local_ratio,abs_log_ratio,covered,uniform
0,5.524328,5.524328,20,0,2.100000e-09,19.981328,False,False
1,10.815557,10.815557,20,3,1.500000e-01,1.897120,False,False
2,4.496078,4.496078,20,6,3.000000e-01,1.203973,True,False
3,13.508791,13.508791,20,26,1.300000e+00,0.262364,True,True
4,11.101747,11.101747,20,1,5.000000e-02,2.995732,False,False


In [ ]:
metrics

CoverageResult(completeness=0.5920714285714286, uniformity=0.4912534684521655, mean_abs_log_ratio=2.3302726975604844, median_abs_log_ratio=1.203972799425936, covered_fraction=0.5920714285714286, n_real=42000, n_fake=42000, k=20, gamma=1.0, tau=0.25, delta=0.6931471805599453)

In [ ]:
ut.save_local_ratio_plot(details, OUT_DIR / f"cfm_local_ratios.png", title=f"cfm: local mass-ratio")

In [ ]:
curve = sample_size_curve(real_z, fake_z, sizes=[1000, 5000, 10000, 25000, 42000], repeats=3, k=20, gamma=1.0, tau=0.25)
curve.groupby("sample_size")[["completeness", "uniformity"]].mean()

,completeness,uniformity
sample_size,,
1000,0.167738,0.000000
5000,0.515683,0.414621
10000,0.451452,0.591024
25000,0.604413,0.502232
42000,0.592071,0.491253


In [ ]:
ut.save_curve_plot(curve, OUT_DIR / f"cfm_sample_size_curve.png")

## Metric-grid

Hyperparameters influence

In [ ]:
def run_metric_grid(
    real_z,
    fake_z,
    ks=(5, 10, 20, 50),
    gammas=(1.0,),
    taus=(0.25,),
    deltas=(math.log(2.0),),
):
    rows = []
    for k in ks:
        for gamma in gammas:
            for tau in taus:
                for delta in deltas:
                    res = mt.estimate_completeness_uniformity(real_z, fake_z, k=k, gamma=gamma, tau=tau, delta=delta)
                    rows.append(asdict(res))
    return pd.DataFrame(rows)

DDIM

In [ ]:
fake_arr = np.load("/content/drive/MyDrive/cursach-hse/ddim_imgs_42000.npy")
fake_z = ut.embed_images(ae, fake_arr, device=device)

grid_df = run_metric_grid(real_z, fake_z, gammas=(0.1, 0.5, 1, 2))

In [ ]:
grid_df

,completeness,uniformity,mean_abs_log_ratio,median_abs_log_ratio,covered_fraction,n_real,n_fake,k,gamma,tau,delta
0,0.000000,NaN,16.985596,16.985596,0.000000,42000,42000,5,0.1,0.25,0.693147
1,0.000095,1.000000,16.983979,16.985596,0.000095,42000,42000,5,0.5,0.25,0.693147
2,0.522167,0.642242,5.181879,0.916291,0.522167,42000,42000,5,1.0,0.25,0.693147
3,0.986214,0.921610,0.228952,0.104089,0.986214,42000,42000,5,2.0,0.25,0.693147
4,0.000000,NaN,16.985596,16.985596,0.000000,42000,42000,10,0.1,0.25,0.693147
5,0.000286,1.000000,16.980760,16.985596,0.000286,42000,42000,10,0.5,0.25,0.693147
6,0.680048,0.637420,2.294187,0.916291,0.680048,42000,42000,10,1.0,0.25,0.693147
7,0.992167,0.926904,0.186809,0.065659,0.992167,42000,42000,10,2.0,0.25,0.693147
8,0.000000,NaN,16.985596,16.985596,0.000000,42000,42000,20,0.1,0.25,0.693147
9,0.000714,1.000000,16.973803,16.985596,0.000714,42000,42000,20,0.5,0.25,0.693147


CFM

In [ ]:
fake_arr = np.load("/content/drive/MyDrive/cursach-hse/cfm_imgs_42000.npy")
fake_z = ut.embed_images(ae, fake_arr, device=device)

grid_df = run_metric_grid(real_z, fake_z, gammas=(0.5, 0.7, 1, 2))

In [ ]:
grid_df

,completeness,uniformity,mean_abs_log_ratio,median_abs_log_ratio,covered_fraction,n_real,n_fake,k,gamma,tau,delta
0,0.000024,1.000000,16.985192,16.985596,0.000024,42000,42000,5,0.5,0.25,0.693147
1,0.004143,0.988506,16.917348,16.985596,0.004143,42000,42000,5,0.7,0.25,0.693147
2,0.323500,0.569957,8.781974,1.609438,0.323500,42000,42000,5,1.0,0.25,0.693147
3,0.976000,0.942281,0.219383,0.097576,0.976000,42000,42000,5,2.0,0.25,0.693147
4,0.000048,1.000000,16.984804,16.985596,0.000048,42000,42000,10,0.5,0.25,0.693147
5,0.011167,0.987207,16.809254,16.985596,0.011167,42000,42000,10,0.7,0.25,0.693147
6,0.450024,0.547061,4.987245,1.609438,0.450024,42000,42000,10,1.0,0.25,0.693147
7,0.988119,0.947302,0.172716,0.072256,0.988119,42000,42000,10,2.0,0.25,0.693147
8,0.000238,1.000000,16.981892,16.985596,0.000238,42000,42000,20,0.5,0.25,0.693147
9,0.030143,0.949447,16.531590,16.985596,0.030143,42000,42000,20,0.7,0.25,0.693147


## Multi-Anchor Distance Distribution

In [ ]:
def local_dimension_slope(sorted_distances, min_rank=5):
    """
    Estimate local intrinsic dimension from log(rank) = d log(distance) + const.
    """
    d = np.asarray(sorted_distances, dtype=np.float64)
    d = d[np.isfinite(d) & (d > 0)]
    if len(d) <= min_rank + 2:
        return float("nan")
    ranks = np.arange(1, len(d) + 1, dtype=np.float64)
    x = np.log(d[min_rank:])
    y = np.log(ranks[min_rank:])
    slope, intercept = np.polyfit(x, y, deg=1)
    return float(slope)


def run_distance_exp(
    real_z,
    fake_z,
    labels=None,
    n_anchors=500,
    max_neighbors=1000,
    seed=42,
):
    """
    For each sampled real anchor:
      1. Find nearest generated anchor.
      2. Compare the first max_neighbors real-real distances and fake-fake distances.
      3. Report local-dimension slopes, KS statistic, Wasserstein distance, and mean-distance ratio.
    """
    rng = np.random.default_rng(seed)
    n_real = len(real_z)
    n_fake = len(fake_z)
    n_anchors = min(n_anchors, n_real)
    anchors = rng.choice(n_real, size=n_anchors, replace=False)

    nn_fake = NearestNeighbors(n_neighbors=1).fit(fake_z)
    _, nearest_fake_idx = nn_fake.kneighbors(real_z[anchors], return_distance=True)
    nearest_fake_idx = nearest_fake_idx[:, 0]

    rr_neighbors = min(max_neighbors + 1, n_real)
    ff_neighbors = min(max_neighbors + 1, n_fake)
    nn_real_all = NearestNeighbors(n_neighbors=rr_neighbors).fit(real_z)
    nn_fake_all = NearestNeighbors(n_neighbors=ff_neighbors).fit(fake_z)

    real_dists, _ = nn_real_all.kneighbors(real_z[anchors], return_distance=True)
    fake_dists, _ = nn_fake_all.kneighbors(fake_z[nearest_fake_idx], return_distance=True)
    # remove self-distance if present as first column
    real_dists = real_dists[:, 1:]
    fake_dists = fake_dists[:, 1:]

    rows = []
    for pos, anchor_idx in enumerate(anchors):
        rd = real_dists[pos]
        fd = fake_dists[pos]
        k_common = min(len(rd), len(fd))
        rd = rd[:k_common]
        fd = fd[:k_common]
        ks = stats.ks_2samp(rd, fd)
        row = {
            "anchor_idx": int(anchor_idx),
            "nearest_fake_idx": int(nearest_fake_idx[pos]),
            "label": int(labels[anchor_idx]) if labels is not None else -1,
            "real_slope": local_dimension_slope(rd),
            "fake_slope": local_dimension_slope(fd),
            "slope_diff": local_dimension_slope(fd) - local_dimension_slope(rd),
            "ks_stat": float(ks.statistic),
            "ks_pvalue": float(ks.pvalue),
            "wasserstein": float(wasserstein_distance(rd, fd)),
            "mean_dist_ratio": float(np.mean(fd) / (np.mean(rd) + 1e-12)),
            "median_dist_ratio": float(np.median(fd) / (np.median(rd) + 1e-12)),
        }
        rows.append(row)
    return pd.DataFrame(rows)

DDIM

In [ ]:
fake_arr = np.load("/content/drive/MyDrive/cursach-hse/ddim_imgs_42000.npy")
fake_z = ut.embed_images(ae, fake_arr, device=device)
ddim_res = run_distance_exp(real_z, fake_z, n_anchors=100, max_neighbors=1000)
ddim_res.describe()

,anchor_idx,nearest_fake_idx,label,real_slope,fake_slope,slope_diff,ks_stat,ks_pvalue,wasserstein,mean_dist_ratio,median_dist_ratio
count,100.000000,100.000000,100.0,100.000000,100.000000,100.000000,100.000000,1.000000e+02,100.000000,100.000000,100.000000
mean,22096.510000,21873.870000,-1.0,12.039205,12.520929,0.481725,0.391750,2.367655e-03,0.818899,1.023651,1.022393
std,11443.907521,12634.146294,0.0,4.507377,4.310937,2.853002,0.208734,1.333133e-02,0.629407,0.089360,0.088562
min,1837.000000,367.000000,-1.0,3.713424,3.957695,-8.919881,0.053000,0.000000e+00,0.052194,0.847020,0.849064
25%,13533.500000,12676.750000,-1.0,8.681906,9.925854,-0.853876,0.237500,6.700349e-129,0.318283,0.972323,0.973871
50%,21788.500000,20667.500000,-1.0,11.906013,12.484384,0.627424,0.369500,7.130152e-61,0.696371,1.014118,1.009309
75%,31986.500000,33857.750000,-1.0,14.725317,15.236641,2.265855,0.530250,2.100750e-22,1.079260,1.063340,1.060319
max,40890.000000,41258.000000,-1.0,30.490045,22.850271,7.096652,0.902000,1.205153e-01,2.595194,1.343084,1.326438


CFM

In [ ]:
fake_arr = np.load("/content/drive/MyDrive/cursach-hse/cfm_imgs_42000.npy")
fake_z = ut.embed_images(ae, fake_arr, device=device)
cfm_res = run_distance_exp(real_z, fake_z, n_anchors=100, max_neighbors=1000)
cfm_res.describe()

,anchor_idx,nearest_fake_idx,label,real_slope,fake_slope,slope_diff,ks_stat,ks_pvalue,wasserstein,mean_dist_ratio,median_dist_ratio
count,100.000000,100.000000,100.0,100.000000,100.000000,100.000000,100.000000,1.000000e+02,100.000000,100.000000,100.000000
mean,22096.510000,19577.900000,-1.0,12.039205,14.067891,2.028687,0.431590,4.993006e-03,0.868702,1.003936,1.000831
std,11443.907521,11970.854578,0.0,4.507377,5.065995,3.317925,0.252897,2.830825e-02,0.691255,0.091269,0.091135
min,1837.000000,105.000000,-1.0,3.713424,4.745092,-5.538006,0.045000,0.000000e+00,0.061840,0.776851,0.775226
25%,13533.500000,9449.500000,-1.0,8.681906,10.644254,0.487016,0.204500,2.696876e-178,0.403783,0.949278,0.948528
50%,21788.500000,19120.000000,-1.0,11.906013,14.045973,1.817685,0.432000,2.921259e-83,0.671552,0.992375,0.988864
75%,31986.500000,29296.500000,-1.0,14.725317,16.947871,3.495038,0.619000,1.122612e-18,1.226087,1.040888,1.034673
max,40890.000000,40977.000000,-1.0,30.490045,32.333023,14.161971,0.957000,2.634717e-01,3.327108,1.345006,1.326526


# Walks in Latent Paths

In [ ]:
def gaussian_to_cube(z: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    return np.clip(ndtr(z), eps, 1.0 - eps)


def cube_to_gaussian(u: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    return ndtri(np.clip(u, eps, 1.0 - eps)).astype(np.float32)


def make_latent_path(
    z0,
    z1,
    T=100,
    family="gaussian_linear",
) -> np.ndarray:
    """
      - gaussian_linear: z_t = (1-t)z0 + t z1
      - cube_linear: u_t = (1-t)Phi(z0) + t Phi(z1), z_t = Phi^{-1}(u_t)
      - slerp: spherical interpolation in flattened Gaussian latent space
    """
    z0 = np.asarray(z0, dtype=np.float32)
    z1 = np.asarray(z1, dtype=np.float32)
    original_shape = z0.shape
    alphas = np.linspace(0.0, 1.0, T, dtype=np.float32)

    if family == "gaussian_linear":
        path = np.stack([(1 - a) * z0 + a * z1 for a in alphas])
    elif family == "cube_linear":
        u0 = gaussian_to_cube(z0)
        u1 = gaussian_to_cube(z1)
        path = np.stack([cube_to_gaussian((1 - a) * u0 + a * u1) for a in alphas])
    elif family == "slerp":
        v0 = z0.reshape(-1).astype(np.float64)
        v1 = z1.reshape(-1).astype(np.float64)
        n0 = np.linalg.norm(v0) + 1e-12
        n1 = np.linalg.norm(v1) + 1e-12
        u0 = v0 / n0
        u1 = v1 / n1
        dot = np.clip(np.dot(u0, u1), -1.0, 1.0)
        omega = np.arccos(dot)
        if omega < 1e-6:
            vecs = np.stack([(1 - a) * v0 + a * v1 for a in alphas])
        else:
            vecs = []
            for a in alphas:
                direction = (np.sin((1 - a) * omega) * u0 + np.sin(a * omega) * u1) / np.sin(omega)
                radius = (1 - a) * n0 + a * n1
                vecs.append(radius * direction)
            vecs = np.stack(vecs)
        path = vecs.reshape((T,) + original_shape).astype(np.float32)
    return path.astype(np.float32)


def path_diagnostics(
    path_z,
    real_z,
    real_labels=None,
    real_support_k=20,
    support_gamma=1.0,
):
    path_z = np.asarray(path_z, dtype=np.float32)
    real_z = np.asarray(real_z, dtype=np.float32)
    T = len(path_z)

    nn1 = NearestNeighbors(n_neighbors=1).fit(real_z)
    nn_dist, nn_idx = nn1.kneighbors(path_z, return_distance=True)
    nn_dist = nn_dist[:, 0]
    nn_idx = nn_idx[:, 0]

    steps = np.linalg.norm(np.diff(path_z, axis=0), axis=1)
    endpoint = np.linalg.norm(path_z[-1] - path_z[0]) + 1e-12
    arc_ratio = float(np.sum(steps) / endpoint)

    if T >= 3:
        curvature = float(np.mean(np.linalg.norm(path_z[2:] - 2 * path_z[1:-1] + path_z[:-2], axis=1)))
    else:
        curvature = float("nan")

    switch_rate = float(np.mean(nn_idx[1:] != nn_idx[:-1]))
    if real_labels is not None:
        nearest_labels = np.asarray(real_labels)[nn_idx]
        class_switch_rate = float(np.mean(nearest_labels[1:] != nearest_labels[:-1]))
    else:
        class_switch_rate = float("nan")

    # Coverage defect: path point is inside the real manifold if it lies inside the adaptive ball
    # of its nearest real neighbor.
    k_eff = min(real_support_k, len(real_z) - 1)
    real_rho = mt.kth_nn_radii(real_z, k=k_eff, exclude_self=True)
    covered_by_real = nn_dist <= support_gamma * real_rho[nn_idx]
    coverage_defect = float(1.0 - np.mean(covered_by_real))

    return {
        "T": int(T),
        "mean_nn_dist": float(np.mean(nn_dist)),
        "max_nn_dist": float(np.max(nn_dist)),
        "switch_rate": switch_rate,
        "class_switch_rate": class_switch_rate,
        "arc_ratio": arc_ratio,
        "curvature": curvature,
        "coverage_defect": coverage_defect,
    }

In [ ]:
z0 = np.random.randn(28 * 28)
z1 = np.random.randn(28 * 28)

gauss_path = make_latent_path(z0, z1, family='gaussian_linear')
cube_path = make_latent_path(z0, z1, family='cube_linear')
slerp_path = make_latent_path(z0, z1, family='slerp')

In [ ]:
np.save('/content/gauss_path.npy', gauss_path)
np.save('/content/cube_path.npy', cube_path)
np.save('/content/slerp_path.npy', slerp_path)

DDIM

In [ ]:
ddim_path_imgs_gauss = np.load('/content/drive/MyDrive/cursach-hse/latent_paths/ddim_gauss_path_imgs.npy')
ddim_path_imgs_cube = np.load('/content/drive/MyDrive/cursach-hse/latent_paths/ddim_cube_path_imgs.npy')
ddim_path_imgs_slerp = np.load('/content/drive/MyDrive/cursach-hse/latent_paths/ddim_slerp_path_imgs.npy')

gauss_emb = ut.embed_images(ae, ddim_path_imgs_gauss, device=device)
cube_emb = ut.embed_images(ae, ddim_path_imgs_cube, device=device)
slerp_emb = ut.embed_images(ae, ddim_path_imgs_slerp, device=device)

In [ ]:
path_diagnostics(gauss_emb, real_z, real_labels=labels, real_support_k=20)

{'T': 100,
 'mean_nn_dist': 9.199965114593505,
 'max_nn_dist': 12.228838920593262,
 'switch_rate': 0.09090909090909091,
 'class_switch_rate': 0.010101010101010102,
 'arc_ratio': 2.553332567214966,
 'curvature': 0.28983163833618164,
 'coverage_defect': 0.020000000000000018}

In [ ]:
path_diagnostics(cube_emb, real_z, real_labels=labels, real_support_k=20)

{'T': 100,
 'mean_nn_dist': 8.249582777023315,
 'max_nn_dist': 10.18299388885498,
 'switch_rate': 0.1414141414141414,
 'class_switch_rate': 0.010101010101010102,
 'arc_ratio': 3.3023388385772705,
 'curvature': 0.47274142503738403,
 'coverage_defect': 0.0}

In [ ]:
path_diagnostics(slerp_emb, real_z, real_labels=labels, real_support_k=20)

{'T': 100,
 'mean_nn_dist': 10.75704761505127,
 'max_nn_dist': 14.27946662902832,
 'switch_rate': 0.030303030303030304,
 'class_switch_rate': 0.010101010101010102,
 'arc_ratio': 2.202319622039795,
 'curvature': 0.16864293813705444,
 'coverage_defect': 0.10999999999999999}

CFM

In [ ]:
cfm_path_imgs_gauss = np.load('/content/drive/MyDrive/cursach-hse/latent_paths/cfm_gauss_path_imgs.npy')
cfm_path_imgs_cube = np.load('/content/drive/MyDrive/cursach-hse/latent_paths/cfm_cube_path_imgs.npy')
cfm_path_imgs_slerp = np.load('/content/drive/MyDrive/cursach-hse/latent_paths/cfm_slerp_path_imgs.npy')

gauss_emb = ut.embed_images(ae, cfm_path_imgs_gauss, device=device)
cube_emb = ut.embed_images(ae, cfm_path_imgs_cube, device=device)
slerp_emb = ut.embed_images(ae, cfm_path_imgs_slerp, device=device)

In [ ]:
path_diagnostics(gauss_emb, real_z, real_labels=labels, real_support_k=20)

{'T': 100,
 'mean_nn_dist': 9.328365244865417,
 'max_nn_dist': 10.87797737121582,
 'switch_rate': 0.12121212121212122,
 'class_switch_rate': 0.030303030303030304,
 'arc_ratio': 3.2934024333953857,
 'curvature': 0.6221853494644165,
 'coverage_defect': 0.21999999999999997}

In [ ]:
path_diagnostics(cube_emb, real_z, real_labels=labels, real_support_k=20)

{'T': 100,
 'mean_nn_dist': 9.009237155914306,
 'max_nn_dist': 9.955808639526367,
 'switch_rate': 0.12121212121212122,
 'class_switch_rate': 0.04040404040404041,
 'arc_ratio': 3.7031118869781494,
 'curvature': 0.7278597950935364,
 'coverage_defect': 0.12}

In [ ]:
path_diagnostics(slerp_emb, real_z, real_labels=labels, real_support_k=20)

{'T': 100,
 'mean_nn_dist': 9.991287460327149,
 'max_nn_dist': 11.946017265319824,
 'switch_rate': 0.09090909090909091,
 'class_switch_rate': 0.030303030303030304,
 'arc_ratio': 3.4002668857574463,
 'curvature': 0.54642254114151,
 'coverage_defect': 0.24}